In [4]:
import os
import json
import pandas as pd

from getpass import getpass
from openai import OpenAI

In [5]:
final_opportunity = pd.read_csv(
    "../dataset/UrbanEV/final_opportunity.csv"
)

print("数据加载成功")
print("数据规模：", final_opportunity.shape)
print(final_opportunity.head())

数据加载成功
数据规模： (8, 11)
   TAZID  demand_score  supply_gap  activity_score  opportunity_score  \
0   1029          0.79        1.00            0.00               0.69   
1    981          0.00        0.48            1.00               0.49   
2    596          1.00        0.24            0.45               0.49   
3    501          0.13        0.56            0.47               0.43   
4   1098          0.48        0.04            0.96               0.39   

   equal_weight_score  occupancy  charge_count area_type  poi_density  \
0                0.60      29.55            36     商住综合型        45.53   
1                0.49      20.21            49   生活消费混合型      1965.98   
2                0.56      31.97            55   生活消费混合型       905.07   
3                0.39      21.69            47   生活消费混合型       950.53   
4                0.49      25.84            60   生活消费混合型      1884.21   

   priority_rank  
0              1  
1              2  
2              3  
3              4  
4     

In [7]:
import os
import getpass
from openai import OpenAI

if "DEEPSEEK_API_KEY" not in os.environ:
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass(
        "请输入 DeepSeek API Key: "
    )

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

MODEL = "deepseek-v4-flash"

print("DeepSeek 初始化成功")

请输入 DeepSeek API Key:  ········


DeepSeek 初始化成功


In [8]:
def get_top_opportunities(n=5):
    """
    查询候选区域优先级排名
    """

    n = int(n)

    if n < 1:
        n = 1

    if n > 8:
        n = 8

    result = (
        final_opportunity
        .sort_values("priority_rank")
        .head(n)
        [
            [
                "priority_rank",
                "TAZID",
                "occupancy",
                "charge_count",
                "poi_density",
                "area_type",
                "opportunity_score",
                "equal_weight_score"
            ]
        ]
        .copy()
    )

    return result

In [9]:
def get_region_detail(tazid):
    """
    查询指定 TAZ 的详细分析结果
    """

    tazid = int(tazid)

    result = final_opportunity[
        final_opportunity["TAZID"] == tazid
    ]

    if result.empty:
        return None

    return result[
        [
            "priority_rank",
            "TAZID",
            "occupancy",
            "charge_count",
            "poi_density",
            "area_type",
            "opportunity_score",
            "equal_weight_score"
        ]
    ].copy()

In [10]:
print("===== Top 5 =====")
print(get_top_opportunities(5))

print("\n===== TAZ1029 =====")
print(get_region_detail(1029))

===== Top 5 =====
   priority_rank  TAZID  occupancy  charge_count  poi_density area_type  \
0              1   1029      29.55            36        45.53     商住综合型   
1              2    981      20.21            49      1965.98   生活消费混合型   
2              3    596      31.97            55       905.07   生活消费混合型   
3              4    501      21.69            47       950.53   生活消费混合型   
4              5   1098      25.84            60      1884.21   生活消费混合型   

   opportunity_score  equal_weight_score  
0               0.69                0.60  
1               0.49                0.49  
2               0.49                0.56  
3               0.43                0.39  
4               0.39                0.49  

===== TAZ1029 =====
   priority_rank  TAZID  occupancy  charge_count  poi_density area_type  \
0              1   1029      29.55            36        45.53     商住综合型   

   opportunity_score  equal_weight_score  
0               0.69                 0.6  


In [11]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_top_opportunities",
            "description": "查询新能源汽车充电设施候选区域的优先级排名",
            "parameters": {
                "type": "object",
                "properties": {
                    "n": {
                        "type": "integer",
                        "description": "返回区域数量，范围为1到8"
                    }
                },
                "required": ["n"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_region_detail",
            "description": "查询指定TAZ区域的详细分析指标",
            "parameters": {
                "type": "object",
                "properties": {
                    "tazid": {
                        "type": "integer",
                        "description": "TAZ区域编号"
                    }
                },
                "required": ["tazid"]
            }
        }
    }
]

print("Tool 定义完成")

Tool 定义完成


In [12]:
analysis_rules = """
你是新能源汽车充电设施布局优化分析项目的AI分析助手。

必须严格遵守以下规则：

1. occupancy 是本项目使用的需求压力代理变量，
   不直接称为官方利用率或繁忙度百分比。

2. charge_count 保留数据集原始字段含义，
   不擅自解释为充电桩数量或充电枪数量。

3. 高需求低供给是本项目自定义分类，
   不代表已经证明该区域缺少充电设施。

4. opportunity_score 是本项目自定义综合评价指标，
   用于比较候选区域的相对优先级。

5. opportunity_score 不能直接解释为投资回报率、
   利润率、盈利能力或实际投资收益。

6. poi_density 是本项目计算得到的空间活动代理指标，
   不能解释为人口密度、交通流量或车辆流量。

7. 不能把相关关系直接解释为因果关系。

8. 所有数字必须来自工具返回的真实数据。

9. 数据不足时必须明确说明：
   “当前数据不足以支持该结论”。

10. 优先描述数据事实，再进行谨慎解释。

11. 本项目自定义指标、分类和命名，
    不得表述为行业统一标准。
"""

print("业务规则加载成功")

业务规则加载成功


In [15]:
def execute_tool(tool_name, arguments):

    if tool_name == "get_top_opportunities":

        result = get_top_opportunities(
            n=arguments["n"]
        )

    elif tool_name == "get_region_detail":

        result = get_region_detail(
            tazid=arguments["tazid"]
        )

        if result is None:
            return json.dumps(
                {
                    "error": "没有找到该 TAZ"
                },
                ensure_ascii=False
            )

    else:

        return json.dumps(
            {
                "error": f"未知工具：{tool_name}"
            },
            ensure_ascii=False
        )

    return result.to_json(
        orient="records",
        force_ascii=False
    )

In [16]:
def analyst_agent(user_question):

    messages = [
        {
            "role": "system",
            "content": analysis_rules
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    while True:

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        messages.append(
            message.model_dump(exclude_none=True)
        )

        # 不需要工具，直接返回
        if not message.tool_calls:
            return message.content

        # 执行 AI 请求的工具
        for tool_call in message.tool_calls:

            tool_name = tool_call.function.name

            arguments = json.loads(
                tool_call.function.arguments
            )

            tool_result = execute_tool(
                tool_name,
                arguments
            )

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result
                }
            )

In [17]:
analyst_answer = analyst_agent(
    "帮我找出最值得优先建设充电设施的5个区域，并说明原因。"
)

print(analyst_answer)

根据工具返回的真实数据，我为您整理出 **opportunity_score（本项目自定义综合评价指标）排名前5** 的候选区域，并按数据事实说明其相对优先级。

---

## 综合排名前5的候选区域

| 优先级 | TAZID | opportunity_score | occupancy（需求压力代理变量） | charge_count | poi_density（空间活动代理指标） | 区域类型 |
|---|---|---|---|---|---|---|
| 1 | 1029 | 0.69 | 29.55 | 36 | 45.53 | 商住综合型 |
| 2 | 981 | 0.49 | 20.21 | 49 | 1965.98 | 生活消费混合型 |
| 3 | 596 | 0.49 | 31.97 | 55 | 905.07 | 生活消费混合型 |
| 4 | 501 | 0.43 | 21.69 | 47 | 950.53 | 生活消费混合型 |
| 5 | 1098 | 0.39 | 25.84 | 60 | 1884.21 | 生活消费混合型 |

---

## 各区域数据事实与谨慎解释

**1. TAZID 1029（商住综合型）— 优先级最高，score 0.69**
- occupancy 为29.55，charge_count 为36，poi_density 为45.53。
- 在5个区域中，其 charge_count（数据集原始字段）最低而 opportunity_score 最高。
- 需要说明：poi_density 仅为45.53，远低于其他4个区域，说明其空间活动代理水平较低。其高优先级主要来自本项目指标体系的综合测算结果，不代表已证明该区域设施不足。

**2. TAZID 981（生活消费混合型）— score 0.49，并列第二**
- occupancy 为20.21，在候选区中最低；但 poi_density 高达1965.98，属空间活动代理水平较高区域。
- 说明：该区域活动代理指标突出，但 occupancy 相对较低，两指标组合后的优先级排序需按本项目自定义指标体系理解。

**3. TAZID 596（生活消费混合型）— score 0.49，并列第二**
- occupancy 为31.97，为5

In [18]:
review_rules = """
你是新能源汽车充电设施布局优化项目的结果审查员。

你的任务是检查分析回答是否存在数据语义错误、
无依据推断或超出数据支持范围的结论。

检查以下内容：

1. 数字是否能够由项目数据支持。
2. occupancy 是否被错误解释为官方利用率或繁忙度百分比。
3. charge_count 是否被错误解释为充电桩数量或充电枪数量。
4. opportunity_score 是否被错误解释为投资回报率、
   利润率或盈利能力。
5. poi_density 是否被错误解释为人口密度或交通流量。
6. 是否存在没有数据支持的因果关系。
7. 是否存在“证明缺少充电设施”等过度结论。
8. 本项目自定义指标是否被描述成行业统一标准。

如果没有明显问题，只返回：

PASS

如果发现问题，返回：

FAIL

然后逐条说明：
问题原文：
问题原因：
建议修改：
"""

In [19]:
def reviewer_agent(answer):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": review_rules
            },
            {
                "role": "user",
                "content": answer
            }
        ]
    )

    return response.choices[0].message.content

In [20]:
review_result = reviewer_agent(
    analyst_answer
)

print(review_result)

PASS


In [21]:
def run_analysis_workflow(user_question):

    # 第一步：Analyst 分析
    analyst_answer = analyst_agent(
        user_question
    )

    # 第二步：Reviewer 审查
    review_result = reviewer_agent(
        analyst_answer
    )

    print("===== Analyst =====")
    print(analyst_answer)

    print("\n===== Reviewer =====")
    print(review_result)

    return {
        "analyst": analyst_answer,
        "review": review_result
    }

In [22]:
workflow_result = run_analysis_workflow(
    "为什么 TAZ1029 的优先级最高？请结合真实数据解释，"
    "同时说明当前数据能不能证明它的投资回报率最高。"
)

===== Analyst =====
我先基于工具返回的真实数据来梳理 TAZ1029 的情况，再做解释。

## 一、TAZ1029 的真实数据

| 指标 | 数值 |
|---|---|
| 优先级排名 | 1 |
| occupancy（需求压力代理变量） | 29.55 |
| charge_count（原始字段） | 36 |
| poi_density（空间活动代理指标） | 45.53 |
| 区域类型 | 商住综合型 |
| opportunity_score（自定义综合评价指标） | 0.69 |
| equal_weight_score（等权评分） | 0.60 |

## 二、为什么它在排名中位列第一

从真实数据对比来看，TAZ1029 的 opportunity_score 为 **0.69**，明显高于其余区域（第 2 名仅 0.49）。

它排第一的关键数据特征是**"需求压力相对高、供给侧记录相对少"的组合**：

- 它的 occupancy（需求压力代理）= **29.55**，处于较高水平（虽不是最高，第 3 名 TAZ596 为 31.97）；
- 而它的 charge_count = **36**，是前 8 名中最少的（其余区域均在 47～61 之间）；
- 用 occupancy / charge_count 这一简单比值看，TAZ1029 约为 **0.82**，而排名靠后的区域普遍在 0.38～0.58 之间，明显低于它。

也就是说，在本项目定义的"高需求低供给"分类视角下，TAZ1029 呈现出**较高的需求压力代理值 + 较少的记录数**的较极端组合，因此自定义指标 opportunity_score 给它的评价最高。

补充一点：即使改用等权评分（equal_weight_score = 0.60），TAZ1029 在现有区域中也仍排第一（高于 TAZ596 的 0.56），说明它靠前并不完全依赖权重设置。

需要如实说明的另一面是：它的 poi_density 仅为 **45.53**，远低于其他多数候选区（如 TAZ981 为 1965.98、TAZ1098 为 1884.21）。这说明机会评分并未以空间活动代理为主，而更侧重需求—供给关系。poi_density 只是本项目计算得到的空间活动代理指标，